# 🚀 Train Startup Survival Agent with Unsloth & TRL

This notebook trains an LLM agent on the **Startup Survival Simulator** — a World Modeling + Long-Horizon Planning environment.

## Environment Mechanics
- **Partial Observability:** `market_demand`, `churn_rate`, and `technical_debt` are hidden. The agent must use `analyze_market` tool to infer them.
- **Sparse Rewards:** 0 per step. Massive milestone payouts at 1k, 2.5k, 5k, 7.5k, 10k users.
- **Mistakes & Recovery:** Rapid hiring builds hidden `technical_debt`. Server Crash at >0.8 debt wipes 20% of users.

## Training Strategy
We generate high-reward rollouts using the expert heuristic, then use **Supervised Fine-Tuning (SFT)** to teach the LLM to imitate successful behavior.

> **Run in Google Colab with a T4 GPU** (Runtime → Change runtime type → T4 GPU)

In [ ]:
# Cell 1: Install dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl>=0.9.0" peft accelerate bitsandbytes datasets
print('Dependencies installed!')

In [ ]:
# Cell 2: Load model with Unsloth 4-bit quantization
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Inject LoRA adapters — only trains ~1% of params
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print(f'Model loaded: {MODEL_NAME}')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# Cell 3: Generate expert trajectories from the live environment
# This runs the expert heuristic against the env to collect high-reward state→action pairs
import sys, json
sys.path.insert(0, '/content')  # Adjust if running locally

# ---- Inline env + expert for Colab (no file uploads needed) ----
import random
from enum import Enum

VALID_ACTIONS = [
    "increase_marketing", "hire_engineer", "improve_product",
    "reduce_costs", "pivot_market", "raise_funding",
    "analyze_market", "refactor_code", "do_nothing"
]

def expert_policy(state: dict, step: int, last_report: dict) -> str:
    """Expert heuristic that knows how to use all tools correctly."""
    cash = state["cash"]
    users = state["users"]
    burn = state["burn_rate"]
    quality = state["product_quality"]
    morale = state["morale"]

    # Tool: check for tech debt warning every 8 steps
    if step > 0 and step % 8 == 0 and cash > 3000:
        return "refactor_code"
    # Tool: analyze market every 5 steps to update world model
    if step > 0 and step % 5 == 0 and cash > 2000:
        return "analyze_market"
    # Survival rules
    if cash < 8000:
        return "reduce_costs"
    if quality < 0.72:
        return "improve_product"
    if users < 500:
        return "increase_marketing"
    if cash > 20000 and burn > (state["revenue"] * 1.2):
        return "raise_funding"
    return "increase_marketing"

print('Expert policy defined.')

In [ ]:
# Cell 4: Build the SFT dataset from expert rollouts
# Each sample = (system_prompt, state_observation) → expert_action
from datasets import Dataset

SYSTEM_PROMPT = """You are an AI CEO managing an early-stage startup.
You observe the current business state and must choose exactly one action.

CRITICAL RULES:
1. Market demand and churn rate are HIDDEN — use analyze_market to estimate them.
2. Hiring engineers builds hidden technical_debt — use refactor_code before it causes a Server Crash.
3. Rewards are SPARSE — you only score points at user milestones (1k, 2.5k, 5k, 7.5k, 10k users).
4. Going bankrupt (cash = 0) gives a -1000 penalty.

Available actions: increase_marketing, hire_engineer, improve_product, reduce_costs,
pivot_market, raise_funding, analyze_market, refactor_code, do_nothing

Reply with ONLY the action name. No explanation."""

# Simulate many episodes with the expert to build dataset
# (We use a simplified inline env for Colab portability)
samples = []

def simulate_episode(seed=42):
    rng = random.Random(seed)
    state = {
        "cash": 50000.0, "users": 100, "revenue": 1000.0,
        "growth_rate": 0.08, "burn_rate": 4500.0,
        "product_quality": 0.55, "morale": 0.7, "time_step": 0
    }
    _market_demand = 0.6
    _churn_rate = 0.03
    _tech_debt = 0.0
    _milestones = set()
    last_report = {}
    episode_samples = []

    for step in range(50):
        action = expert_policy(state, step, last_report)
        episode_samples.append({"state": dict(state), "action": action, "step": step})

        # Apply action effects (simplified)
        if action == "increase_marketing":
            state["burn_rate"] += 1200; state["growth_rate"] = min(0.35, state["growth_rate"] + 0.03)
            _market_demand = min(1.0, _market_demand + 0.03)
        elif action == "hire_engineer":
            state["burn_rate"] += 2200; state["product_quality"] = min(1.0, state["product_quality"] + 0.07)
            _tech_debt += 0.15
        elif action == "improve_product":
            state["product_quality"] = min(1.0, state["product_quality"] + 0.05)
            _churn_rate = max(0.01, _churn_rate - 0.006)
        elif action == "reduce_costs":
            state["burn_rate"] = max(1800, state["burn_rate"] - 1600)
            state["growth_rate"] = max(0.02, state["growth_rate"] - 0.015)
        elif action == "raise_funding":
            prob = min(0.85, 0.2 + state["product_quality"] * 0.3 + min(0.35, state["users"] / 5000))
            if rng.random() < prob:
                state["cash"] += 30000
        elif action == "analyze_market":
            state["cash"] = max(0, state["cash"] - 1000)
            last_report = {"estimated_demand": round(_market_demand + rng.uniform(-0.05, 0.05), 2),
                           "estimated_churn": round(_churn_rate + rng.uniform(-0.01, 0.01), 3),
                           "tech_debt_warning": _tech_debt > 0.6}
        elif action == "refactor_code":
            state["cash"] = max(0, state["cash"] - 2500)
            _tech_debt = max(0.0, _tech_debt - 0.4)

        # Market noise
        _market_demand = max(0.2, min(1.0, _market_demand + rng.uniform(-0.02, 0.02)))
        _churn_rate = max(0.01, min(0.25, _churn_rate + rng.uniform(-0.003, 0.003)))

        # Server crash
        if _tech_debt > 0.8:
            state["users"] = int(state["users"] * 0.8)
            state["morale"] = max(0.2, state["morale"] - 0.3)
            _tech_debt = 0.4

        # Growth
        acquired = int(state["users"] * state["growth_rate"] * (0.7 + state["product_quality"] * 0.6)
                        * (0.6 + _market_demand * 0.8) * (0.75 + state["morale"] * 0.35))
        if state["users"] < 150: acquired += 25
        lost = int(state["users"] * _churn_rate)
        state["users"] = max(0, state["users"] + acquired - lost)

        arpu = 14.0 + state["product_quality"] * 6.0
        state["revenue"] = round(state["users"] * arpu * max(0.35, _market_demand), 2)
        state["cash"] = round(max(0.0, state["cash"] + state["revenue"] - state["burn_rate"]), 2)
        state["time_step"] = step + 1

        if state["cash"] <= 0 or state["users"] >= 10000:
            break

    return episode_samples, state["users"]

# Generate 50 episodes with different seeds
all_samples = []
successful_episodes = 0
for seed in range(50):
    ep_samples, final_users = simulate_episode(seed=seed)
    if final_users > 500:  # Only include episodes that made meaningful progress
        all_samples.extend(ep_samples)
        successful_episodes += 1

print(f'Generated {len(all_samples)} samples from {successful_episodes}/50 successful episodes.')

In [ ]:
# Cell 5: Format into chat template
EOS_TOKEN = tokenizer.eos_token

def format_sample(sample):
    state = sample["state"]
    action = sample["action"]
    step = sample["step"]

    user_msg = f"""Step {step}. Current startup state:
- Cash: ${state['cash']:,.0f}
- Users: {state['users']:,}
- Revenue/step: ${state['revenue']:,.0f}
- Burn rate/step: ${state['burn_rate']:,.0f}
- Product quality: {state['product_quality']:.2f}
- Team morale: {state['morale']:.2f}
- Time step: {state['time_step']}

Note: Market demand, churn rate, and technical debt are hidden — use tools to discover them.
What action do you take?"""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": action},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text + EOS_TOKEN}

formatted = [format_sample(s) for s in all_samples]
dataset = Dataset.from_list(formatted)
dataset = dataset.train_test_split(test_size=0.05, seed=42)

print(f'Train: {len(dataset["train"])} samples | Val: {len(dataset["test"])} samples')
print(f'\nSample prompt preview:')
print(dataset["train"][0]["text"][:500])

In [ ]:
# Cell 6: Train with SFTTrainer
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=120,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        eval_steps=20,
        save_steps=60,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

print('Starting training...')
trainer_stats = trainer.train()
print(f'Training complete! Runtime: {trainer_stats.metrics["train_runtime"]:.1f}s')

In [ ]:
# Cell 7: Plot training loss curve
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
steps = [x["step"] for x in log_history if "loss" in x]
losses = [x["loss"] for x in log_history if "loss" in x]

plt.figure(figsize=(8, 4))
plt.plot(steps, losses, color='#1a73e8', linewidth=2)
plt.fill_between(steps, losses, alpha=0.1, color='#1a73e8')
plt.title('SFT Training Loss — Startup Survival Agent', fontsize=14)
plt.xlabel('Training Step', fontsize=12)
plt.ylabel('Cross-Entropy Loss', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150)
plt.show()
print(f'Final loss: {losses[-1]:.4f}')

In [ ]:
# Cell 8: Save LoRA adapters
model.save_pretrained("startup_agent_lora")
tokenizer.save_pretrained("startup_agent_lora")
print('LoRA adapters saved to ./startup_agent_lora')

# (Optional) Push to Hugging Face Hub
# import os
# os.environ["HF_TOKEN"] = "hf_xxxxxxxxxxxx"  # replace with your token
# model.push_to_hub("your-username/startup-agent-lora", token=os.environ["HF_TOKEN"])
# tokenizer.push_to_hub("your-username/startup-agent-lora", token=os.environ["HF_TOKEN"])
# print('Model pushed to HuggingFace Hub!')

In [ ]:
# Cell 9: Quick sanity test — run the trained model on a sample state
FastLanguageModel.for_inference(model)

test_state = {
    "cash": 22000.0, "users": 450, "revenue": 4200.0,
    "burn_rate": 8500.0, "product_quality": 0.68,
    "morale": 0.65, "time_step": 15
}

user_msg = f"""Step 15. Current startup state:
- Cash: $22,000 (WARNING: low)
- Users: 450
- Revenue/step: $4,200
- Burn rate/step: $8,500 (EXCEEDS revenue!)
- Product quality: 0.68
- Team morale: 0.65
- Time step: 15

Note: Market demand, churn rate, and technical debt are hidden — use tools to discover them.
What action do you take?"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_msg},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True,
                                        add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=16, temperature=0.1, do_sample=True)
response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()

print(f'State: cash=$22k, burn>revenue, users=450')
print(f'Expert would choose: reduce_costs')
print(f'Trained model chose: {response}')
print(f'Match: {response == "reduce_costs"}')